<a href="https://colab.research.google.com/github/christine-a11y/cassava-leaf-disease-classification/blob/main/notebooks/eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

KGAT_498fe2f0248b52a16a932ae1dd051954

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
import hashlib


from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset
import torchvision.transforms.v2 as T
from torch.utils.data import DataLoader

In [ ]:
kaggle_data = {
    "username": "christinemkhitaryan",
    "key": "KGAT_0f4aed1533144c0613c9545d74416a12"
}

# 2. Ստեղծում ենք kaggle.json-ը
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_data, f)

!chmod 600 /root/.kaggle/kaggle.json
print("✅ Kaggle Token-ը տեղադրվեց:")

In [ ]:
import os

# Տեղադրեք Kaggle-ից պատճենած API Token-ը այստեղ
os.environ['KAGGLE_API_TOKEN'] = "KGAT_498fe2f0248b52a16a932ae1dd051954"

print("✅ Token-ը ակտիվացված է:")

In [ ]:
# 3. Ներբեռնում ենք ֆայլը
!kaggle competitions download -c cassava-leaf-disease-classification

# 4. Ապափաթեթավորում ենք cassava_data թղթապանակում
!unzip -q cassava-leaf-disease-classification.zip -d cassava_data

# 5. Ջնջում ենք zip-ը, որ տեղ ազատվի
!rm cassava-leaf-disease-classification.zip

In [ ]:
# Տեսնում ենք նկարների լեյբլները (CSV-ն)
df = pd.read_csv('cassava_data/train.csv')
print("Տվյալների քանակը:", len(df))
print(df.head())

# Տեսնում ենք հիվանդությունների անվանումները
with open('cassava_data/label_num_to_disease_map.json') as f:
    mapping = json.load(f)
print("\nՀիվանդությունների ցանկը:", mapping)

In [ ]:
df

Starting EDA

In [ ]:
print(len(df))
print(df.head())


In [ ]:
print("=== 1.  ԸՆԴՀԱՆՈՒՐ ՏԵՂԵԿՈՒԹՅՈՒՆ ===")
df.info()

In [ ]:
print("\n=== 2. MISSING VALUES  ===")
missing_data = df.isnull().sum()
print(missing_data)

0: Cassava Bacterial Blight (CBB)

1: Cassava Brown Streak Disease (CBSD)

2: Cassava Green Mottle (CGM)

3: Cassava Mosaic Disease (CMD)

4: Healthy

Unique values

In [ ]:
unique_labels = sorted(df['label'].unique())
print("Unique labels in dataset:", unique_labels)

Unique classes-ի քանակը

In [ ]:
num_classes = df['label'].nunique()
print("Number of unique classes:", num_classes)

Գտնում ենք բոլոր այն արժեքները, որոնք 0-4 range-ից դուրս են

In [ ]:
expected_set = {0, 1, 2, 3, 4}
invalid_rows = df[~df['label'].isin(expected_set)]
print(f"Invalid label rows count: {len(invalid_rows)}")

In [ ]:
df_viz = df.copy()

label_map = {
    0: "Cassava Bacterial Blight (CBB)",
    1: "Cassava Brown Streak Disease (CBSD)",
    2: "Cassava Green Mottle (CGM)",
    3: "Cassava Mosaic Disease (CMD)",
    4: "Healthy"
}

#  Ավելացնում ենք հիվանդության անվան սյունյակը copy-ի մեջ
df_viz['disease_name'] = df_viz['label'].map(label_map)

df_viz

Յուրքանչյուր տեսակից քանի նկար կա

In [ ]:
class_counts = df_viz['disease_name'].value_counts()
print(class_counts)

Նույնը տոկոսային հարաբերությամբ

In [ ]:
class_percentages = (df_viz['disease_name'].value_counts(normalize=True).sort_index() * 100).round(2)

print("--- Class Percentages ---")
print(class_percentages)

Կառուցենք  Bar Chart

In [ ]:
chart_data = df_viz['disease_name'].value_counts().reset_index()
chart_data.columns = ['disease_name', 'count']
chart_data['percentage'] = (chart_data['count'] / len(df_viz) * 100).round(2)

# 2. Կառուցում ենք գրաֆիկը
plt.figure(figsize=(10, 5))
ax = sns.barplot(
    data=chart_data,
    x='disease_name',
    y='count',
    palette='viridis'
)

plt.title('Class Distribution (df_viz)', fontsize=13, fontweight='bold')
plt.xlabel('Disease', fontsize=11)
plt.ylabel('Image Count', fontsize=11)
plt.xticks(rotation=20, ha='right')

# 3. Ավելացնում ենք քանակն ու տոկոսը սյուների վրա
for p in ax.patches:
    height = int(p.get_height())
    pct = (height / len(df_viz) * 100)
    ax.annotate(
        f'{height}\n({pct:.1f}%)',
        (p.get_x() + p.get_width() / 2., height),
        ha='center', va='bottom',
        fontsize=9, xytext=(0, 3),
        textcoords='offset points'
    )

plt.tight_layout()
plt.show()

In [ ]:
#Գտնենք ամենամեծ class-ը
max_class_name = class_counts.idxmax()
max_class_count = class_counts.max()
max_class_pct = (max_class_count / len(df_viz)) * 100

# 2. Գտնենք ամենափոքր class-ը
min_class_name = class_counts.idxmin()
min_class_count = class_counts.min()
min_class_pct = (min_class_count / len(df_viz)) * 100

# 3. Համեմատենք class-երի չափերը (Imbalance Ratio)
imbalance_ratio = max_class_count / min_class_count

print(f" Ամենամեծ class:  {max_class_name} -> {max_class_count:,} պատկեր ({max_class_pct:.2f}%)")
print(f" Ամենափոքր class: {min_class_name} -> {min_class_count:,} պատկեր ({min_class_pct:.2f}%)")
print(f" Imbalance ratio (Max / Min): {imbalance_ratio:.2f}x")

Խիստ անհամաչափություն (Class Imbalance):
Ամենամեծ դասը (CMD) ~12.1 անգամ ավելի մեծ է, քան ամենափոքր դասը (CBB)։

Dominant Class:
Ամբողջ դատասեթի պատկերների ավելի քան 61%-ը պատկանում է ընդամենը 1 դասի (CMD)։

Եթե մոդելը սովորական ձևով թրեյն անենք, այն կարող է բիասավորվել (biased) դեպի CMD-ն և վատ ճանաչել CBB-ն կամ մյուս հազվադեպ դասերը։ Սա նշանակում է, որ հետագայում պետք կգա․

Stratified K-Fold (որպեսզի ամեն fold-ում դասերի հարաբերակցությունը պահպանվի)

Weighted Loss Function կամ Class Weights

Հատուկ Augmentation տեխնիկաներ փոքր դասերի համա

**Դիտարկենք յուրաքանչյուր class-ից sample-ներ **

In [ ]:
print("Ա ընթացիկ թղթապանակի ֆայլերը:")
print(os.listdir('.'))

if os.path.exists('cassava_data/train_images'):
    IMAGES_DIR = 'cassava_data/train_images'
else:
    IMAGES_DIR = 'cassava_data'

print(f"Օգտագործվող path: {IMAGES_DIR}")

In [ ]:
IMAGES_DIR = 'cassava_data/train_images'

if not os.path.exists(IMAGES_DIR):
    print(f"⚠️ Թղթապանակը չի գտնվել: {IMAGES_DIR}")
    print("Խնդրում եմ ստուգել ճիշտ path-ը os.listdir()-ով:")


samples_per_class = 3
fig, axes = plt.subplots(5, samples_per_class, figsize=(15, 15))

for label in range(5):
    class_df = df_viz[df_viz['label'] == label].sample(samples_per_class)
    disease_title = class_df['disease_name'].iloc[0]

    for i, (_, row) in enumerate(class_df.iterrows()):
        img_path = os.path.join(IMAGES_DIR, row['image_id'])
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        ax = axes[label, i]
        ax.imshow(img)
        ax.axis('off')
        if i == 1:
            ax.set_title(f"Class {label}: {disease_title}", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

CBB (Class 0): Տերևների վրա առկա են մուգ շագանակագույն, չորացած անկյունաձև բծեր (water-soaked lesions) և եզրերի այրված/չորացած տեսք։

CBSD (Class 1): Դեղնավուն և շագանակագույն նախշերը տարածված են հիմնականում տերևի ջղերի (veins) երկայնքով։

CGM (Class 2): Տերևի մակերեսին նկատվում են մանր, սպեկտրալ դեղնավուն/բաց-կանաչավուն կետեր ու բծեր (mottling)։

CMD (Class 3): Տերևները խիստ ծռմռված են, կնճռոտված և ասիմետրիկ՝ դեղնականաչավուն խճանկարային pattern-ով։

Healthy (Class 4): Տերևները հիմնականում հարթ են և առողջ կանաչ, սակայն առկա է Label Noise (որոշ նկարներում տեսանելի են հիվանդության բծեր կամ արտաքին վնասվածքներ)։

Նմանություններ և Ռիսկեր: CBB-ն և CBSD-ն տեսողականորեն բավականին նման են (շագանակագույն բծեր/դեղնում), ինչը կարող է առաջացնել misclassification: Background-ները և լուսավորվածությունը խիստ փոփոխական են։

Վերջին շարքի աջ կողմի երրորդ նկարում ակնհայտ երևում են չորացած, շագանակագույն բծեր, որոնք CBB-ի կամ CBSD-ի ախտանիշ են, բայց դատասեթում այն նշված է որպես Class 4 (Healthy)։

Cassava Leaf Disease դատասեթի ամենահայտնի խնդիրներից մեկն է՝ Label Noise (սխալ/աղմկոտ լեյբլավորում)։

Մոդելը չի կարող հասնել 100% Accuracy-ի․ Քանի որ ground truth (ճիշտ պատասխան համարվող) տվյալների մեջ սխալներ կան, նույնիսկ կատարյալ մոդելը Kaggle-ում ~89-90% accuracy-ից վերև դժվար է բարձրանում։

Label Smoothing Loss․ Սովորական Cross-Entropy Loss-ի փոխարեն շատ օգտակար է օգտագործել Label Smoothing (օրինակ՝ smoothing=0.1)։ Սա թույլ է տալիս, որ մոդելը 100% վստահությամբ չհավատա սխալ լեյբլին և պատժվի ավելի քիչ, երբ տվյալներում աղմուկ (noise) կա։

Symmetric Cross Entropy (SCE) կամ Bi-Tempered Loss․ Այս հատուկ Loss function-ները նախագծված են հենց այսպիսի աղմկոտ (noisy) դատասեթերի դեմ պայքարելու համար։

Background-ների տարբերությունը

Խիստ բազմազան են․ Նկարներում երևում են հող, այլ ոչ հիվանդ/առողջ բույսեր, չորացած ցողուններ, իսկ որոշ sample-ներում նաև մարդու ձեռքեր կամ տերևը պահող մատներ։

Մոդելի ռիսկ․ Մոդելը կարող է պատահաբար սովորել background-ի դետալները (օրինակ՝ հողի գույնը) տերևի հիվանդության հետ կապելու փոխարեն։

Լուծում․ Պարտադիր է Random Resized Crop և Zoom augmentation-ների օգտագործումը, որպեսզի մոդելի ուշադրությունը կենտրոնանա միայն տերևի վրա։

Lighting-ի (լուսավորվածության) տարբերությունը

Խիստ փոփոխական է․ Կան պայծառ, ուղիղ արևի տակ արված նկարներ (որոնք տերևի վրա սպիտակ փայլեր/glare են առաջացնում), ինչպես նաև ստվերում կամ մռայլ եղանակին նկարված մութ կադրեր։

Մոդելի ռիսկ․ Արևի փայլերը մոդելը կարող է սխալմամբ ընդունել որպես CGM-ի դեղնավուն բծեր, իսկ մութ ստվերները՝ CBB-ի չորացած հատվածներ։

Լուծում․ ColorJitter, Brightness, and Contrast augmentation-ների կիրառումը թրեյնինգի ընթացքում։

In [ ]:
image_shapes = []
formats = set()

# Անցնում ենք բոլոր նկարների վրայով և հավաքում տվյալները
for img_name in df_viz['image_id']:
    img_path = os.path.join(IMAGES_DIR, img_name)
    with Image.open(img_path) as img:
        width, height = img.size
        # Get channels (e.g. RGB -> 3)
        channels = len(img.getbands())
        formats.add(img.format)
        image_shapes.append((width, height, channels))

# Դարձնում ենք DataFrame վերլուծության համար
shapes_df = pd.DataFrame(image_shapes, columns=['width', 'height', 'channels'])

print("--- Image Dimensions & Channels ---")
print(shapes_df.describe())
print(f"\nUnique Formats: {formats}")
print(f"Unique Channels: {shapes_df['channels'].unique()}")

In [ ]:
widths, heights = [], []

# Օգտագործում ենք ավելի վաղ ստացված shapes_df-ը կամ նորից կարդում sample
sample_df = df_viz.sample(1000, random_state=42)
for img_name in sample_df['image_id']:
    img_path = os.path.join(IMAGES_DIR, img_name)
    img = cv2.imread(img_path)
    if img is not None:
        heights.append(img.shape[0])
        widths.append(img.shape[1])

plt.figure(figsize=(6, 6))
plt.scatter(widths, heights, alpha=0.5, c='blue')
plt.title("Image Width vs Height")
plt.xlabel("Width (pixels)")
plt.ylabel("Height (pixels)")
plt.grid(True)
plt.show()

Սա վերջնականապես վստահեցնում է, որ preprocessing-ի ժամանակ բարդ aspect-ratio-preserving resize-ների (օրինակ՝ եզրերին սև գոտիներ / padding ավելացնելուն) կարիք չկա։ Կարող ենք ուղիղ resize անել քառակուսի (օրինակ՝ 256x256)։

Արդյունքները

Width & Height: Բոլոր պատկերները standard 800 x 600 resolution ունեն (800px width, 600px height)։

Minimum Dimensions: 800 x 600

Maximum Dimensions: 800 x 600

Channels: 3 (RGB color channels)

Image Format: JPEG (.jpg)

Key Takeaway Modeling-ի համար.
Դատասեթը չափսերի առումով լիովին uniform (միատարր) է՝ չկան տարբեր չափսերի կամ corrupted format-ով նկարներ։ Այնուամենայնիվ, թրեյնինգի ժամանակ բոլորը resize կանենք (օրինակ՝ 224x224, 384x384 կամ 512x512), որպեսզի GPU-ն ավելի արագ մշակի։

In [ ]:
# 1. Missing / Invalid labels & Duplicate rows
missing_labels = df['label'].isnull().sum()
invalid_labels = (~df['label'].isin(range(5))).sum()
duplicate_rows = df.duplicated().sum()

# 2. Missing files, Corrupted images, Formats & Image Hashing for Duplicates
missing_files = 0
corrupted_images = 0
unexpected_formats = []
image_hashes = set()
duplicate_images_count = 0

for img_name in df['image_id']:
    img_path = os.path.join(IMAGES_DIR, img_name)

    # Check if file exists
    if not os.path.exists(img_path):
        missing_files += 1
        continue

    try:
        with Image.open(img_path) as img:
            img.verify() # Corrupted image check

        # Format check
        if not img_name.lower().endswith(('.jpg', '.jpeg')):
            unexpected_formats.append(img_name)

        # Duplicate images check via MD5 Hash
        with open(img_path, 'rb') as f:
            file_hash = hashlib.md5(f.read()).hexdigest()
            if file_hash in image_hashes:
                duplicate_images_count += 1
            else:
                image_hashes.add(file_hash)

    except Exception:
        corrupted_images += 1

print("=== Data Quality & Integrity Report ===")
print(f" Missing labels: {missing_labels}")
print(f" Invalid labels (not 0-4): {invalid_labels}")
print(f" Duplicate DataFrame rows: {duplicate_rows}")
print(f" Missing image files: {missing_files}")
print(f" Corrupted images: {corrupted_images}")
print(f" Duplicate images (exact file match): {duplicate_images_count}")
print(f" Unexpected file formats: {len(unexpected_formats)}")

Դատասեթը տեխնիկական առումով շատ մաքուր է (ֆայլերը ամբողջական են, չկան corrupted կամ չբացվող նկարներ): Միակ հիմնական խնդիրները, որոնց դեմ պետք է պայքարենք մոդելավորման ժամանակ, նախորդ քայլերում գտած Class Imbalance-ը և Label Noise-ն են

In [ ]:
colors = ('b', 'g', 'r')
plt.figure(figsize=(15, 10))

for label in range(5):
    class_sample = df_viz[df_viz['label'] == label].sample(100, random_state=42)
    disease_name = class_sample['disease_name'].iloc[0]

    mean_hist = {c: np.zeros((256, 1)) for c in colors}

    for img_name in class_sample['image_id']:
        img_path = os.path.join(IMAGES_DIR, img_name)
        img = cv2.imread(img_path)
        for i, col in enumerate(colors):
            hist = cv2.calcHist([img], [i], None, [256], [0, 256])
            mean_hist[col] += hist / 100

    plt.subplot(2, 3, label + 1)
    for col in colors:
        plt.plot(mean_hist[col], color=col)
    plt.title(disease_name)
    plt.xlim([0, 256])

plt.tight_layout()
plt.show()

Blue Channel Spike (մուգ կապույտ պիկը 0-ի վրա)․
Բոլոր դասերի մոտ 0-ական ինտենսիվության վրա կա շատ բարձր Blue (B) պիկ։ Սա ցույց է տալիս, որ նկարներում Blue channel-ի ցածր արժեքները (գրեթե 0) շատ են, ինչը բնական է կանաչ տերևների և մուգ/ստվերոտ ֆոնի համար (RGB համակարգում կանաչն ու դեղինը ունեն շատ ցածր Blue ինտենսիվություն)։

Green vs Red channels (Կանաչի և Կարմիրի հարաբերակցությունը)․

Healthy (Առողջ)․ Կանաչ (Green) կորը 150-200 տիրույթում ակնհայտ բարձր է Կարմիրից (Red)։ Սա մաթեմատիկորեն հաստատում է առողջ, հագեցած կանաչ տերևների առկայությունը։

CBB & CBSD (Հիվանդություններ)․ Կանաչի և Կարմիրի կորերը շատ մոտ են իրար, իսկ որոշ հատվածներում Կարմիրը բարձրանում է։ Սա չորացած, շագանակագույն ու դեղնած հատվածների (Green-ի նվազման և Red-ի բարձրացման) ուղղակի հետևանքն է։

CGM & CMD․ Ունեն բավականին հավասարաչափ բաշխում, քանի որ տերևների վրա կան թե՛ կանաչ, թե՛ դեղնավուն (Mosaic/Mottle) հատվածներ։

Color Augmentations-ի սահմանափակում․
Hue (գույնի երանգ) augmentation-ը պետք է կիրառել շատ զգույշ կամ շատ փոքր range-ով (օրինակ՝ hue_shift_limit=5): Եթե գույները շատ փոխենք, կանաչ տերևը կարող է դառնալ դեղնավուն, և մոդելը Healthy-ն կշփոթի CGM-ի կամ CBSD-ի հետ։

Brightness / Contrast Augmentation-ի անհրաժեշտություն․
Քանի որ 0-250 տիրույթում ինտենսիվությունները լայն տարածված են (լուսավորության խիստ տատանումների պատճառով), Brightness/Contrast փոփոխությունները կօգնեն մոդելին անկախ լինել լուսավորությունից։

Normalization Constants (ImageNet standards)․
Թրեյնինգի ժամանակ ImageNet-ի standard mean/std-ով normalize անելը (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) լիովին արդարացված է, քանի որ RGB ալիքների բաշխվածությունը բնական լուսանկարների standard կառուցվածք ունի։

In [ ]:
plt.figure(figsize=(15, 6))

for label in range(5):
    class_sample = df_viz[df_viz['label'] == label].sample(100, random_state=42)
    disease_name = class_sample['disease_name'].iloc[0]

    # Նախապես ստեղծում ենք զրոյական մատրիցա
    mean_img = np.zeros((600, 800, 3), dtype=np.float32)

    for img_name in class_sample['image_id']:
        img_path = os.path.join(IMAGES_DIR, img_name)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mean_img += img / 100

    mean_img = np.uint8(mean_img)

    plt.subplot(1, 5, label + 1)
    plt.imshow(mean_img)
    plt.title(f"Class {label}")
    plt.axis('off')

plt.tight_layout()
plt.show()

Կենտրոնացված լուսավորություն/դեղնություն (Class 0 & Class 1)՝
Class 0-ի (CBB) և Class 1-ի (CBSD) միջինացված պատկերներում կենտրոնում ակնհայտ երևում է ավելի վառ, դեղնավուն/բաց-կանաչ կլորավուն տիրույթ։ Սա հաստատում է, որ այս հիվանդությունների դեպքում ֆոտոգրաֆները/ֆերմերները նկարելիս կադրի կենտրոնը ուղղել են հենց վնասված, դեղնած տերևի վրա։

Լղոզված և միատարր ֆոն (Class 2, 3, 4)՝
Class 2-ի, 3-ի և 4-ի (Healthy) մոտ պատկերը շատ ավելի միատարր կանաչ-մոխրագույն է։ Սա նշանակում է, որ այս դասերում տերևների դիրքավորումն ու չափսերը կադրում ավելի պատահական են բաշխված։

Ոչ մի class-ի մոտ չկա հստակ «տերևի տեսք» (edges): Սա ցույց է տալիս, որ դատասեթում տերևների անկյունները (orientations), մասշտաբները (scales) և background-ները խիստ բազմազան են։

Spatial/Position Invariance-ի կարիք՝
Քանի որ հիվանդությունները միայն կենտրոնում չեն, մոդելը չպետք է «կախվածություն» ձեռք բերի նկարի կենտրոնական հատվածից։

Անհրաժեշտ Augmentation-ներ՝

Random Horizontal & Vertical Flips: Օգնում է, որ մոդելը սովորի pattern-ները անկախ տերևի ուղղությունից։

ShiftScaleRotate / Random Crop: Ստիպում է մոդելին գտնել ախտանիշները, նույնիսկ եթե տերևը կադրի անկյունում է կամ խոշոր պլանով (zoom):

## **Preprocessing**

In [ ]:
# 1. Train/Validation Split (80% Train, 20% Val)
# Stratified split-ը պահպանում է class-երի տոկոսային հարաբերակցությունը
train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df['label']
)

# 2. Ստուգել set-երի չափերը
print(f"Training set size: {len(train_df)} images ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation set size: {len(val_df)} images ({len(val_df)/len(df)*100:.1f}%)")




# 3. Համեմատել Class Distribution-ը երկու set-երում (Տոկոսային)
train_dist = train_df['label'].value_counts(normalize=True).sort_index() * 100
val_dist = val_df['label'].value_counts(normalize=True).sort_index() * 100

dist_df = pd.DataFrame({
    'Train (%)': train_dist,
    'Validation (%)': val_dist
})
print("\n--- Class Distribution Comparison ---")
print(dist_df.round(2))

# 4. Համոզվել, որ validation images-ը training-ում չեն հայտնվել (Data Leakage Check)
intersection = set(train_df['image_id']).intersection(set(val_df['image_id']))
print(f"\nOverlap between Train and Validation: {len(intersection)} images")
assert len(intersection) == 0, "ERROR: Data leakage detected!"

In [ ]:
IMG_SIZE = 384

# 1. PyTorch / torchvision Preprocessing Pipelines
def get_pytorch_transforms(img_size=IMG_SIZE):
    train_transform = T.Compose([
        # 1. Spatial & Geometry Augmentations
        T.RandomResizedCrop(size=(img_size, img_size), scale=(0.8, 1.0)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),


        # 2. Color & Lighting Augmentations (Արև/Ստվեր սիմուլյացիա)
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),

        # 3. Tensor Formatting & Normalization
        T.PILToTensor(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    valid_transform = T.Compose([
        T.Resize(size=(img_size, img_size)),
        T.PILToTensor(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    return train_transform, valid_transform

# 2. PyTorch Dataset Class (օգտագործելով PIL Image-ներ)
class CassavaPyTorchDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir # Պահում ենք նկարների պապկայի path-ը
        self.image_ids = self.df['image_id'].values # Օգտագործում ենք օրիգինալ 'image_id'-ն
        self.labels = self.df['label'].values
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Ֆայլի path-ը կառուցում ենք տեղում
        img_path = os.path.join(self.img_dir, self.image_ids[idx])

        # Կարդում ենք նկարը
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label


IMAGES_DIR ='cassava_data/train_images'

train_tr, _ = get_pytorch_transforms()

# dataset-ին տալիս ենք օրիգինալ train_df-ն ու IMAGES_DIR-ը
sample_dataset = CassavaPyTorchDataset(train_df, img_dir=IMAGES_DIR, transform=train_tr)

img_tensor, label_tensor = sample_dataset[0]

print(f"Tensor Shape: {img_tensor.shape}") # [3, 384, 384]
print(f"Label: {label_tensor}")




#Augmentation-ի Ստուգման Կոդ (Visual Verification)

In [ ]:
def get_visual_augmentation_pipeline(img_size=384):
    return T.Compose([
        T.RandomResizedCrop(size=(img_size, img_size), scale=(0.8, 1.0)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        T.RandomRotation(degrees=15),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1)
    ])


def check_augmentation_quality(df, img_dir, num_samples=5, augmentations_per_img=3):
    aug_pipeline = get_visual_augmentation_pipeline()

    # Ընտրում ենք տարբեր դասերի 3 նկար
    sample_rows = df.drop_duplicates(subset=['label']).head(num_samples)

    fig, axes = plt.subplots(num_samples, augmentations_per_img + 1, figsize=(16, 4 * num_samples))

    for row_idx, (_, row) in enumerate(sample_rows.iterrows()):
        img_path = os.path.join(img_dir, row['image_id'])
        label = row['label']
        orig_img = Image.open(img_path).convert("RGB")

        # Original Image
        axes[row_idx, 0].imshow(orig_img.resize((384, 384)))
        axes[row_idx, 0].set_title(f"ORIGINAL (Label: {label})", color='green', fontweight='bold')
        axes[row_idx, 0].axis('off')

        # Augmented Variants
        for aug_idx in range(1, augmentations_per_img + 1):
            aug_img = aug_pipeline(orig_img)
            axes[row_idx, aug_idx].imshow(aug_img)
            axes[row_idx, aug_idx].set_title(f"Augmented #{aug_idx}")
            axes[row_idx, aug_idx].axis('off')

    plt.tight_layout()
    plt.show()

# 3. Աշխատացնել վիզուալիզացիան
check_augmentation_quality(train_df, img_dir=IMAGES_DIR)

Preprocessing & Data Pipeline Specification1. Data Ingestion & Lazy Loading Strategy

Image Loading Mechanics:Պատկերները ֆայլային համակարգից (Disk/SSD) կարդացվում են ըստ պահանջի (Lazy Loading)՝ կանխելու RAM-ի գերբեռնումը։ Օգտագործվում է PIL.Image.open(), որից անմիջապես հետո կատարվում է .convert("RGB")՝ երաշխավորելով 3-ալիքանի RGB ձևաչափը (արտահոսքերը կամ RGBA/Grayscale անոմալիաները բացառելու համար)։


Leak-Free Dataset Design: CassavaPyTorchDataset դասը չի փոփոխում սկզբնական DataFrame-ը։ Ֆայլերի հասցեները կառուցվում են դինամիկ կերպով (os.path.join(img_dir, image_id)), ինչը ապահովում է ճկունություն միջավայրերի միջև (Colab, Local GPU, Kaggle):


2. Spatial Transformations & Resolution

Target Resolution (384x384): Սկզբնական 800x600 չափսից իջեցումը 384x384-ի օպտիմալ բալանս է GPU VRAM-ի ծախսի և տերևների մանր ախտանիշների (նեկրոտիկ բծեր, ջղերի դեղնացում) պահպանման միջև։

Training Resizing Strategy (RandomResizedCrop):size=(384, 384), scale=(0.8, 1.0)Պատահականորեն կտրում է պատկերի 80-100%-ը և մասշտաբավորում։ Սա կանխում է մոդելի կապվածությունը ախտանիշների կոնկրետ չափսի կամ դիրքի հետ (Scale Invariance):

Validation Resizing Strategy (Resize):size=(384, 384)Ուղղակի Resize՝ առանց պատահական կտրումների։ Սա ապահովում է Validation Metric-ների դետերմինիստիկ, կրկնելի և ճշգրիտ գնահատումը։


3. Data Augmentations (Regularization)

Overfitting-ից խուսափելու և մոդելի generalizability-ն բարձրացնելու համար Train pipeline-ում կիրառվում են հետևյալ սպատիալ ձևափոխությունները՝

RandomHorizontalFlip(p=0.5) — 50% հավանականությամբ հորիզոնական արտացոլում։

RandomVerticalFlip(p=0.5) — 50% հավանականությամբ ուղղահայաց արտացոլում։

4. Tensor Conversion & Intensity Normalization


PILToTensor(): PIL Image պատկերը փոխարկում է PyTorch Tensor-ի (uint8), միաժամանակ փոխելով չափսերի կարգը՝ [Height, Width, Channels] $\rightarrow$ [Channels, Height, Width] ([3, 384, 384])։

ToDtype(torch.float32, scale=True): Փիքսելների ինտենսիվության արժեքները [0, 255] integer տիրույթից նորմալիզացվում են [0.0, 1.0] float32 միջակայքի։

Normalize(mean, std): Կիրառվում է ImageNet-ի ստանդարտ վեկտորային նորմալիզացիան՝

mean = [0.485, 0.456, 0.406]

std = [0.229, 0.224, 0.225]

Տեխնիկական հիմնավորում․ Ամեն RGB ալիքի արժեքները բերվում են [-2.1, 2.6] տիրույթի (զրոյական միջինով և միավոր դիսպերսիայով)։ Սա անհրաժեշտ է pre-trained backbone-ների (EfficientNet, ViT) ճիշտ weights-երի օգտագործման և gradient vanishing/explosion-ը կանխելու համար։



### ** Data Loader **

DataLoader-ի 4 Հիմնական Ֆունկցիաները


Batching (Խմբավորում)․
Նեյրոնային ցանցերը սովորում են ոչ թե 1 նկար նայելով, այլ նկարների խմբերով (օրինակ՝ 16 կամ 32 հատ)։ DataLoader-ը անհատական նկարները հավաքում է 1 միասնական Matrix Tensor-ի մեջ ([16, 3, 384, 384])։

Parallel Loading & No GPU Bottleneck (num_workers)․
Մինչ GPU-ն հաշվարկում է ընթացիկ 16 նկարների gradient-ները, DataLoader-ը CPU-ի միջուկների միջոցով ֆոնային ռեժիմում արդեն կարդում ու transform է անում հաջորդ 16 նկարները։ GPU-ն երբեք պարապուրդի մեջ չի սպասում CPU-ին։

Memory Optimization (Lazy Loading + pin_memory)․
Այն RAM-ի մեջ չի պահում բոլոր 21,000 նկարները։ Կարդում է միայն այդ պահին անհրաժեշտ 16 նկարը և pin_memory=True-ի շնորհիվ դրանք վայրկենապես տեղափոխում է GPU VRAM-ի մեջ։

Data Shuffling (shuffle=True)․
Ամեն Epoch-ի սկզբում խառնում է նկարների հերթականությունը, որպեսզի մոդելը չհիշի «թե որ նկարից հետո որն է գալիս», այլ սովորի հենց տերևի հիվանդության հատկանիշները։

In [ ]:
BATCH_SIZE = 16
NUM_WORKERS = 2  # Colab-ի CPU-ի միջուկների քանակը

# 2. Ստանում ենք մեր սահմանած Transforms-ը
train_transform, valid_transform = get_pytorch_transforms(img_size=384)

# 3. Ստեղծում ենք Dataset-ները
train_dataset = CassavaPyTorchDataset(
    df=train_df,
    img_dir=IMAGES_DIR,
    transform=train_transform
)

valid_dataset = CassavaPyTorchDataset(
    df=val_df,
    img_dir=IMAGES_DIR,
    transform=valid_transform
)

# 4. Ստեղծում ենք DataLoader-ները
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,        # Train-ի ժամանակ խառնում ենք, որ մոդելը հերթականություն չանգիրի
    num_workers=NUM_WORKERS,
    pin_memory=True,     # Արագացնում է Tensor-ների տեղափոխումը CPU-ից GPU
    drop_last=True       # Վերջին անկատար batch-ը դեն ենք նետում (որ batch norm-ը չխափանվի)
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,       # Validation-ի ժամանակ shuffle պետք չէ
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)

# 5. Sanity Check / Ստուգում առաջին Batch-ով
images_batch, labels_batch = next(iter(train_loader))

print(f"Batch Images Shape : {images_batch.shape}")  # [16, 3, 384, 384]
print(f"Batch Labels Shape : {labels_batch.shape}")  # [16]
print(f"Batch Image Type   : {images_batch.dtype}")  # torch.float32
print(f"Batch Label Type   : {labels_batch.dtype}")